# DEA
---
_**D**ata **E**nvelopment **A**nalysis_

In [1]:
import numpy as np
import pandas as pd

from scipy.optimize import linprog

## 1. Introduction
---
**DEA** (_Data Envelopment Analysis_) evaluates the **relative efficiency** of comparable **decision-making units** (DMUs) that convert multiple inputs into multiple outputs.

This notebook implements the **input-oriented CCR model**, named after Charnes, Cooper and Rhodes (1978), assuming **constant returns to scale** (CRS). It finds the largest proportional input reduction compatible with maintaining at least the observed outputs, then checks remaining **slacks**. The result supports benchmarking; several units can be efficient simultaneously.

### 1.1. Applications
---
The _DEA method_ can support comparisons of similar operating units, such as:

- Recycling centers
- Production lines
- Service branches
- Distribution depots

Consider **8 recycling centers** operating with comparable sorting technology and incoming material quality. For the same month, record **labor hours** and **electricity** (kWh) as inputs, and saleable **paper** and **plastic** (tonnes) as outputs. Greater output and lower input consumption are preferred.

All data are **illustrative**. CRS assumes that activities can be scaled proportionally over the modeled operating range. These small peer sets make the calculations transparent; practical studies need enough comparable units and relevant variables to distinguish performance meaningfully.

In [2]:
df = pd.DataFrame({
    "labor hours": [400, 600, 500, 750, 400, 800, 600, 1200],
    "electricity": [6000, 4000, 5000, 7500, 8000, 10000, 6000, 12000],
    "paper": [40, 20, 35, 35, 40, 30, 30, 70],
    "plastic": [20, 40, 35, 35, 20, 30, 35, 70]
})

df.index = [f'Center {name}' for name in "ABCDEFGH"]

df.style.background_gradient()

,labor hours,electricity,paper,plastic
Center A,400,6000,40,20
Center B,600,4000,20,40
Center C,500,5000,35,35
Center D,750,7500,35,35
Center E,400,8000,40,20
Center F,800,10000,30,30
Center G,600,6000,30,35
Center H,1200,12000,70,70


## 2. Method steps
---

### 2.1. Inputs and outputs
---
Separate the input matrix $X$ and output matrix $Y$. In the DataFrames, rows are DMUs and columns are variables. The mathematical notation below uses $x_{ij}$ for input $i$ of unit $j$, and $y_{rj}$ for output $r$ of unit $j$.

Use original physical quantities, rather than preference weights or min-max scores. This recipe requires **strictly positive**, finite observations. Choose variables that describe the same production process for every unit.

In [3]:
INPUTS = ["labor hours", "electricity"]
OUTPUTS = ["paper", "plastic"]

X = df[INPUTS].astype(float)
Y = df[OUTPUTS].astype(float)

assert np.isfinite(df.to_numpy()).all() and (df > 0).all().all()

pd.DataFrame({
    "role": ["input", "input", "output", "output"],
    "unit": ["hours/month", "kWh/month", "tonnes/month", "tonnes/month"]
}, index=INPUTS + OUTPUTS)

,role,unit
labor hours,input,hours/month
electricity,input,kWh/month
paper,output,tonnes/month
plastic,output,tonnes/month


### 2.2. Input-oriented CCR model
---
For the evaluated unit $o$, solve the **envelopment model**:

$$ \large
\begin{aligned}
\min\quad &\theta\\
\text{subject to}\quad
&\sum_{j=1}^{n}\lambda_jx_{ij}\leq\theta x_{io}, &&i=1,\ldots,m,\\
&\sum_{j=1}^{n}\lambda_jy_{rj}\geq y_{ro}, &&r=1,\ldots,s,\\
&\lambda_j\geq0,\quad\theta\geq0.
\end{aligned}
$$

The nonnegative intensities $\lambda_j$ construct a reference activity from the observations. **There is no constraint $\sum_j\lambda_j=1$**: that would introduce variable returns to scale, changing the model.

With the evaluated unit included in the peer set, $0<\theta^*\leq1$. The quantity $1-\theta^*$ is the possible **radial reduction** of every input, before additional slack adjustments.

### 2.3. Linear programming coefficients
---
Evaluate **Center F** first. Use the variable order $(\lambda_1,\ldots,\lambda_n,\theta)$. SciPy requires inequalities of the form $Az\leq b$, so negate the output constraints.

Each DMU column in the table below is the coefficient of its corresponding $\lambda_j$.

In [4]:
FOCUS = "Center F"

n = len(X)
m = len(INPUTS)
s = len(OUTPUTS)
x_o = X.loc[FOCUS].to_numpy()
y_o = Y.loc[FOCUS].to_numpy()

c = np.r_[np.zeros(n), 1]
A_ub = np.vstack([
    np.c_[X.to_numpy().T, -x_o],
    np.c_[-Y.to_numpy().T, np.zeros(s)]
])
b_ub = np.r_[np.zeros(m), -y_o]

df_lp = pd.DataFrame(
    A_ub, index=INPUTS + OUTPUTS, columns=list(df.index) + ["theta"]
)
df_lp["right-hand side"] = b_ub

df_lp

,Center A,Center B,Center C,Center D,Center E,Center F,Center G,Center H,theta,right-hand side
labor hours,400.0,600.0,500.0,750.0,400.0,800.0,600.0,1200.0,-800.0,0.0
electricity,6000.0,4000.0,5000.0,7500.0,8000.0,10000.0,6000.0,12000.0,-10000.0,0.0
paper,-40.0,-20.0,-35.0,-35.0,-40.0,-30.0,-30.0,-70.0,0.0,-30.0
plastic,-20.0,-40.0,-35.0,-35.0,-20.0,-30.0,-35.0,-70.0,0.0,-30.0


### 2.4. Radial efficiency
---
Minimize the final variable using SciPy's **HiGHS** solver. Check success before interpreting the solution. The radial target multiplies every observed input by the same efficiency factor.

In [5]:
result = linprog(
    c, A_ub=A_ub, b_ub=b_ub,
    bounds=[(0, None)]*(n + 1), method="highs"
)

if not result.success:
    raise RuntimeError(result.message)

theta = float(result.x[-1])

pd.Series({
    "success": result.success,
    "radial efficiency": theta,
    "radial input reduction (%)": 100*(1 - theta)
})

success                            True
radial efficiency              0.535714
radial input reduction (%)    46.428571
dtype: object

In [6]:
pd.DataFrame({
    "observed input": X.loc[FOCUS],
    "radial target": theta*X.loc[FOCUS]
})

,observed input,radial target
labor hours,800.0,428.571429
electricity,10000.0,5357.142857


### 2.5. Remaining slacks
---
Fix $\theta=\theta^*$ and identify remaining **input excesses** $s_i^-$ and **output surpluses** $s_r^+$:

$$ \large
\begin{aligned}
\max\quad
&\sum_{i=1}^{m}\frac{s_i^-}{x_{io}}+
\sum_{r=1}^{s}\frac{s_r^+}{y_{ro}}\\
\text{subject to}\quad
&\sum_j\lambda_jx_{ij}+s_i^-=\theta^*x_{io},\\
&\sum_j\lambda_jy_{rj}-s_r^+=y_{ro},\\
&\lambda_j,s_i^-,s_r^+\geq0.
\end{aligned}
$$

This second phase preserves the optimal radial score. Positive, dimensionless slack coefficients provide a **unit-invariant choice of projection**; they do not turn this diagnostic into a new DEA efficiency score. If several projections maximize the objective, their peer combinations can differ.

Use variable order $(\lambda,s^-,s^+)$. Negate the objective to maximize with `linprog`.

In [7]:
c_slack = np.r_[np.zeros(n), -1/x_o, -1/y_o]

A_eq = np.block([
    [X.to_numpy().T, np.eye(m), np.zeros((m, s))],
    [Y.to_numpy().T, np.zeros((s, m)), -np.eye(s)]
])
b_eq = np.r_[theta*x_o, y_o]

result_slack = linprog(
    c_slack, A_eq=A_eq, b_eq=b_eq,
    bounds=[(0, None)]*(n + m + s), method="highs"
)

if not result_slack.success:
    raise RuntimeError(result_slack.message)

LAMBDAS = pd.Series(result_slack.x[:n], index=df.index)
INPUT_SLACK = pd.Series(result_slack.x[n:n + m], index=INPUTS)
OUTPUT_SLACK = pd.Series(result_slack.x[n + m:], index=OUTPUTS)

pd.concat({"input excess": INPUT_SLACK, "output surplus": OUTPUT_SLACK})

input excess    labor hours       0.000000
                electricity    1071.428571
output surplus  paper            -0.000000
                plastic          -0.000000
dtype: float64

### 2.6. Peers and benchmark targets
---
Positive intensities identify the **peers** contributing to this projection. Intensities express activity scale, rather than probabilities or criteria weights.

$$ \large
\widehat{x}_{io}=\sum_j\lambda_jx_{ij}=\theta^*x_{io}-s_i^-,
\qquad
\widehat{y}_{ro}=\sum_j\lambda_jy_{rj}=y_{ro}+s_r^+
$$

In [8]:
TOLERANCE = 1e-8

LAMBDAS[LAMBDAS > TOLERANCE].to_frame("reference intensity")

,reference intensity
Center C,0.857143


In [9]:
TARGET_INPUTS = X.T @ LAMBDAS
TARGET_OUTPUTS = Y.T @ LAMBDAS

assert np.allclose(TARGET_INPUTS + INPUT_SLACK, theta*X.loc[FOCUS])
assert np.allclose(TARGET_OUTPUTS - OUTPUT_SLACK, Y.loc[FOCUS])

pd.DataFrame({
    "observed input": X.loc[FOCUS],
    "radial target": theta*X.loc[FOCUS],
    "additional input slack": INPUT_SLACK,
    "final target": TARGET_INPUTS
})

,observed input,radial target,additional input slack,final target
labor hours,800.0,428.571429,0.000000,428.571429
electricity,10000.0,5357.142857,1071.428571,4285.714286


In [10]:
pd.DataFrame({
    "observed output": Y.loc[FOCUS],
    "output surplus": OUTPUT_SLACK,
    "benchmark output": TARGET_OUTPUTS
})

,observed output,output surplus,benchmark output
paper,30.0,-0.0,30.0
plastic,30.0,-0.0,30.0


Center F has radial efficiency approximately **0.5357**, allowing a common input reduction of **46.43%**. Its reference is **6/7 of Center C**. After radial reduction, electricity has an additional excess of approximately **1,071.43 kWh**.

The final benchmark uses about **428.57 labor hours** and **4,285.71 kWh** for the same **30 tonnes of each output**. The example separates proportional contraction from an additional improvement in one resource.

### 2.7. Repeat for every unit
---
Repeat the same two linear programs for each DMU. The following helper reuses the calculations above and returns the radial score, intensities and slacks.

A unit is **strongly CCR efficient** only when $\theta^*=1$ and the maximum attainable slack objective is zero. A radial score of 1 alone is insufficient. Use relative slacks and a numerical tolerance for this check.

In [11]:
def ccr_input(X, Y, unit):
    n, m = X.shape
    s = Y.shape[1]
    x_o = X.loc[unit].to_numpy(dtype=float)
    y_o = Y.loc[unit].to_numpy(dtype=float)

    radial = linprog(
        np.r_[np.zeros(n), 1],
        A_ub=np.vstack([
            np.c_[X.to_numpy().T, -x_o],
            np.c_[-Y.to_numpy().T, np.zeros(s)]
        ]),
        b_ub=np.r_[np.zeros(m), -y_o],
        bounds=[(0, None)]*(n + 1), method="highs"
    )
    if not radial.success:
        raise RuntimeError(radial.message)
    theta = float(radial.x[-1])

    slack = linprog(
        np.r_[np.zeros(n), -1/x_o, -1/y_o],
        A_eq=np.block([
            [X.to_numpy().T, np.eye(m), np.zeros((m, s))],
            [Y.to_numpy().T, np.zeros((s, m)), -np.eye(s)]
        ]),
        b_eq=np.r_[theta*x_o, y_o],
        bounds=[(0, None)]*(n + m + s), method="highs"
    )
    if not slack.success:
        raise RuntimeError(slack.message)

    return (
        theta,
        pd.Series(slack.x[:n], index=X.index),
        pd.Series(slack.x[n:n + m], index=X.columns),
        pd.Series(slack.x[n + m:], index=Y.columns)
    )


RESULTS = {unit: ccr_input(X, Y, unit) for unit in X.index}

THETA = pd.Series({unit: solution[0] for unit, solution in RESULTS.items()})
PEERS = pd.DataFrame({unit: solution[1] for unit, solution in RESULTS.items()}).T
INPUT_SLACKS = pd.DataFrame({unit: solution[2] for unit, solution in RESULTS.items()}).T
OUTPUT_SLACKS = pd.DataFrame({unit: solution[3] for unit, solution in RESULTS.items()}).T

relative_slack = (INPUT_SLACKS/X).sum(axis=1) + (OUTPUT_SLACKS/Y).sum(axis=1)

df_efficiency = pd.DataFrame({
    "radial efficiency": THETA,
    "radial reduction (%)": 100*(1 - THETA).clip(lower=0),
    "strongly efficient": (abs(THETA - 1) <= TOLERANCE) & (relative_slack <= TOLERANCE)
})

df_efficiency.style.background_gradient(subset=["radial efficiency"])

,radial efficiency,radial reduction (%),strongly efficient
Center A,1.000000,0.000000,True
Center B,1.000000,0.000000,True
Center C,1.000000,0.000000,True
Center D,0.666667,33.333333,False
Center E,1.000000,0.000000,False
Center F,0.535714,46.428571,False
Center G,0.833333,16.666667,False
Center H,0.833333,16.666667,False


In [12]:
pd.concat({"input excess": INPUT_SLACKS, "output surplus": OUTPUT_SLACKS}, axis=1)

input excess              output surplus              
          labor hours  electricity          paper       plastic
Center A          0.0    -0.000000            0.0 -0.000000e+00
Center B         -0.0     0.000000           -0.0  0.000000e+00
Center C         -0.0    -0.000000           -0.0  0.000000e+00
Center D         -0.0     0.000000           -0.0 -0.000000e+00
Center E          0.0  2000.000000            0.0 -4.689582e-14
Center F          0.0  1071.428571           -0.0 -0.000000e+00
Center G         -0.0     0.000000            5.0  0.000000e+00
Center H         -0.0     0.000000           -0.0 -0.000000e+00

In [13]:
# Show only reference columns that contribute to at least one projection.
PEERS.loc[:, (PEERS > TOLERANCE).any()].style.background_gradient()

,Center A,Center B,Center C
Center A,1.000000,0.000000,-0.000000
Center B,0.000000,1.000000,-0.000000
Center C,0.000000,0.000000,1.000000
Center D,0.000000,0.000000,1.000000
Center E,1.000000,0.000000,-0.000000
Center F,0.000000,0.000000,0.857143
Center G,0.000000,-0.000000,1.000000
Center H,0.000000,0.000000,2.000000


**Centers A, B and C** are strongly efficient under these observations. Center D can match C's outputs with **one-third less of each input**. Center E has a radial score of **1**, yet its **2,000 kWh** electricity excess makes it **not strongly efficient**. Center G's benchmark also produces **5 extra tonnes of paper**.

Center H uses the reference activity **2 × Center C**, illustrating the CRS assumption and why the intensities need not sum to one. These findings identify different improvement opportunities; they do not impose a total ranking on efficient units or on equal radial scores.

## 3. Examples
---

### 3.1. Bakery production shifts
---
Compare **6 production shifts** making the same two bread products with common recipes and quality requirements. Inputs are **flour** (kg) and **labor** (hours); outputs are saleable **loaves** and **rolls** (units). Equipment and operating conditions are assumed comparable, and CRS is retained over this production range.

The following illustrative example shows how a benchmark can combine **two peers**. Unlike a preference-based product ranking, DEA asks whether the observed production could be achieved with fewer resources.

In [14]:
df_ex1 = pd.DataFrame({
    "flour": [200, 240, 220, 330, 250, 360],
    "labor": [40, 30, 35, 52.5, 50, 50],
    "loaves": [500, 400, 550, 550, 500, 600],
    "rolls": [1000, 1600, 1500, 1500, 1000, 2000]
})

df_ex1.index = [f'Shift {name}' for name in "ABCDEF"]

df_ex1.style.background_gradient()

,flour,labor,loaves,rolls
Shift A,200,40.000000,500,1000
Shift B,240,30.000000,400,1600
Shift C,220,35.000000,550,1500
Shift D,330,52.500000,550,1500
Shift E,250,50.000000,500,1000
Shift F,360,50.000000,600,2000


In [15]:
INPUTS_ex1 = ["flour", "labor"]
OUTPUTS_ex1 = ["loaves", "rolls"]

X_ex1 = df_ex1[INPUTS_ex1].astype(float)
Y_ex1 = df_ex1[OUTPUTS_ex1].astype(float)

assert np.isfinite(df_ex1.to_numpy()).all() and (df_ex1 > 0).all().all()

# Solve the radial and slack phases for each shift.
RESULTS_ex1 = {unit: ccr_input(X_ex1, Y_ex1, unit) for unit in X_ex1.index}

THETA_ex1 = pd.Series({unit: solution[0] for unit, solution in RESULTS_ex1.items()})
PEERS_ex1 = pd.DataFrame({unit: solution[1] for unit, solution in RESULTS_ex1.items()}).T
INPUT_SLACKS_ex1 = pd.DataFrame({unit: solution[2] for unit, solution in RESULTS_ex1.items()}).T
OUTPUT_SLACKS_ex1 = pd.DataFrame({unit: solution[3] for unit, solution in RESULTS_ex1.items()}).T

relative_slack_ex1 = (
    (INPUT_SLACKS_ex1/X_ex1).sum(axis=1) + (OUTPUT_SLACKS_ex1/Y_ex1).sum(axis=1)
)

df_efficiency_ex1 = pd.DataFrame({
    "radial efficiency": THETA_ex1,
    "radial reduction (%)": 100*(1 - THETA_ex1).clip(lower=0),
    "strongly efficient": (
        (abs(THETA_ex1 - 1) <= TOLERANCE) & (relative_slack_ex1 <= TOLERANCE)
    )
})

df_efficiency_ex1.style.background_gradient(subset=["radial efficiency"])

,radial efficiency,radial reduction (%),strongly efficient
Shift A,1.000000,0.000000,False
Shift B,1.000000,0.000000,True
Shift C,1.000000,0.000000,True
Shift D,0.666667,33.333333,False
Shift E,0.800000,20.000000,False
Shift F,0.828571,17.142857,False


In [16]:
pd.concat({"input excess": INPUT_SLACKS_ex1, "output surplus": OUTPUT_SLACKS_ex1}, axis=1)

input excess           output surplus            
               flour     labor         loaves       rolls
Shift A     0.000000  8.181818           -0.0  363.636364
Shift B    -0.000000  0.000000           -0.0    0.000000
Shift C    -0.000000 -0.000000            0.0   -0.000000
Shift D    -0.000000  0.000000           -0.0   -0.000000
Shift E     0.000000  8.181818           -0.0  363.636364
Shift F     1.142857  0.000000           -0.0    0.000000

In [17]:
PEERS_ex1.loc[:, (PEERS_ex1 > TOLERANCE).any()].style.background_gradient()

,Shift B,Shift C
Shift A,0.000000,0.909091
Shift B,1.000000,-0.000000
Shift C,0.000000,1.000000
Shift D,0.000000,1.000000
Shift E,0.000000,0.909091
Shift F,0.714286,0.571429


In [18]:
FOCUS_ex1 = "Shift F"
lambda_ex1 = PEERS_ex1.loc[FOCUS_ex1]

target_inputs_ex1 = X_ex1.T @ lambda_ex1
target_outputs_ex1 = Y_ex1.T @ lambda_ex1

pd.DataFrame({
    "observed input": X_ex1.loc[FOCUS_ex1],
    "radial target": THETA_ex1[FOCUS_ex1]*X_ex1.loc[FOCUS_ex1],
    "additional input slack": INPUT_SLACKS_ex1.loc[FOCUS_ex1],
    "final target": target_inputs_ex1
})

,observed input,radial target,additional input slack,final target
flour,360.0,298.285714,1.142857,297.142857
labor,50.0,41.428571,0.000000,41.428571


In [19]:
pd.DataFrame({
    "observed output": Y_ex1.loc[FOCUS_ex1],
    "output surplus": OUTPUT_SLACKS_ex1.loc[FOCUS_ex1],
    "benchmark output": target_outputs_ex1
})

,observed output,output surplus,benchmark output
loaves,600.0,-0.0,600.0
rolls,2000.0,0.0,2000.0


**Shifts B and C** are strongly efficient. Shift F has a radial efficiency of approximately **0.8286**. Its reference combines **5/7 of B** and **4/7 of C**, producing the same **600 loaves and 2,000 rolls** with about **297.14 kg of flour** and **41.43 labor hours**. The flour target includes an additional **1.14 kg** slack reduction after radial contraction.

The benchmark is a modeled activity assembled from observed practices, rather than a staffing or production schedule. Its usefulness depends on the comparability of the shifts, selected inputs and outputs, and returns-to-scale assumption.

## 4. References
---

- Charnes, A., Cooper, W. W. and Rhodes, E. (1978). [*Measuring the efficiency of decision making units*](https://doi.org/10.1016/0377-2217(78)90138-8).
- Cooper, W. W., Seiford, L. M. and Tone, K. (2007). [*Data Envelopment Analysis: A Comprehensive Text with Models, Applications, References and DEA-Solver Software*](https://doi.org/10.1007/978-0-387-45283-8). Chapters 2–3 discuss CCR efficiency, slacks and projections.
- Ji, Y.-b. and Lee, C. (2010). [*Data envelopment analysis*](https://doi.org/10.1177/1536867X1001000207). Sections 2 and 4 illustrate radial efficiency, slacks and two-phase calculations.